# PyDeequ Pre-Validation: raw_schema_layer_table & silver_schema_layer_table
Creates two Delta tables with the **same column names** and runs simple pre-validation data quality checks on both.

In [0]:
%pip install pydeequ==1.3.0
%restart_python

  Obtaining dependency information for pydeequ==1.3.0 from https://files.pythonhosted.org/packages/e5/3b/7ebaa95cf3a1510dc2262f775703afe33f787a78d0b57f01f8eb93441fcc/pydeequ-1.3.0-py3-none-any.whl.metadata
  Using cached pydeequ-1.3.0-py3-none-any.whl.metadata (9.5 kB)
Using cached pydeequ-1.3.0-py3-none-any.whl (37 kB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%env SPARK_VERSION=3.1

env: SPARK_VERSION=3.1


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult
from pydeequ.analyzers import AnalysisRunner, AnalyzerContext, Size

spark._jvm.com.amazon.deequ.checks.CheckLevel.Error()
print("Deequ JVM classes found.")

TARGET_SCHEMA = "asper_production_asper_us_dp_dev_catalog.test_atul"
RAW_TABLE     = f"{TARGET_SCHEMA}.pre_val_raw_schema_layer_table"
SILVER_TABLE  = f"{TARGET_SCHEMA}.pre_val_silver_schema_layer_table"

Deequ JVM classes found.


## 1. Common schema (both tables share the same column names)

In [0]:
SCHEMA = T.StructType([
    T.StructField("customer_id",   T.StringType(),  True),
    T.StructField("customer_name", T.StringType(),  True),
    T.StructField("email",         T.StringType(),  True),
    T.StructField("country",       T.StringType(),  True),
    T.StructField("order_amount",  T.DoubleType(),  True),
    T.StructField("order_date",    T.DateType(),    True),
    T.StructField("status",        T.StringType(),  True),
])
EXPECTED_COLUMNS = [f.name for f in SCHEMA.fields]

## 2. Create raw_schema_layer_table (contains intentional data issues)

In [0]:
from datetime import date

raw_data = [
    ("C001", "Alice",  "alice@example.com",  "IN", 250.0,  date(2026, 1, 5),  "ACTIVE"),
    ("C002", "Bob",    None,                 "US", 120.5,  date(2026, 1, 6),  "ACTIVE"),
    ("C003", None,     "charlie@example",    "GB", -40.0,  date(2026, 1, 7),  "INACTIVE"),  # null name, bad email, negative amount
    ("C003", "Charlie","charlie@example.com","GB", 75.0,   date(2026, 1, 7),  "INACTIVE"),  # duplicate id
    (None,   "Diana",  "diana@example.com",  "XX", 300.0,  None,              "UNKNOWN"),   # null id, bad country/status, null date
    ("C006", "Eve",    "eve@example.com",    "IN", 999999.0, date(2026, 1, 9), "ACTIVE"),   # outlier amount
]
raw_df = spark.createDataFrame(raw_data, SCHEMA)

raw_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(RAW_TABLE)
print(f"Created {RAW_TABLE}")
display(spark.table(RAW_TABLE))

Created asper_production_asper_us_dp_dev_catalog.test_atul.pre_val_raw_schema_layer_table


customer_id,customer_name,email,country,order_amount,order_date,status
C003,Charlie,charlie@example.com,GB,75.0,2026-01-07,INACTIVE
C001,Alice,alice@example.com,IN,250.0,2026-01-05,ACTIVE
C002,Bob,null,US,120.5,2026-01-06,ACTIVE
C003,null,charlie@example,GB,-40.0,2026-01-07,INACTIVE
null,Diana,diana@example.com,XX,300.0,null,UNKNOWN
C006,Eve,eve@example.com,IN,999999.0,2026-01-09,ACTIVE


## 3. Create silver_schema_layer_table (clean data)

In [0]:
silver_data = [
    ("C001", "Alice",   "alice@example.com",   "IN", 250.0, date(2026, 1, 5), "ACTIVE"),
    ("C002", "Bob",     "bob@example.com",     "US", 120.5, date(2026, 1, 6), "ACTIVE"),
    ("C003", "Charlie", "charlie@example.com", "GB", 75.0,  date(2026, 1, 7), "INACTIVE"),
    ("C004", "Diana",   "diana@example.com",   "US", 300.0, date(2026, 1, 8), "ACTIVE"),
    ("C005", "Eve",     "eve@example.com",     "IN", 180.0, date(2026, 1, 9), "ACTIVE"),
]
silver_df = spark.createDataFrame(silver_data, SCHEMA)

silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TABLE)
print(f"Created {SILVER_TABLE}")
display(spark.table(SILVER_TABLE))

Created asper_production_asper_us_dp_dev_catalog.test_atul.pre_val_silver_schema_layer_table


customer_id,customer_name,email,country,order_amount,order_date,status
C003,Charlie,charlie@example.com,GB,75.0,2026-01-07,INACTIVE
C004,Diana,diana@example.com,US,300.0,2026-01-08,ACTIVE
C005,Eve,eve@example.com,IN,180.0,2026-01-09,ACTIVE
C001,Alice,alice@example.com,IN,250.0,2026-01-05,ACTIVE
C002,Bob,bob@example.com,US,120.5,2026-01-06,ACTIVE


## 4. Helpers

In [0]:
ALL_RESULTS = []

def run_check(df: DataFrame, check: Check, test_name: str, table: str):
    result = VerificationSuite(spark).onData(df).addCheck(check).run()
    rdf = VerificationResult.checkResultsAsDataFrame(spark, result)
    print(f"\n{'=' * 70}\n{test_name} [{table}] -> {result.status}\n{'=' * 70}")
    display(rdf.select("constraint", "constraint_status", "constraint_message"))
    ALL_RESULTS.append(rdf.withColumn("test", F.lit(test_name)).withColumn("table", F.lit(table)))
    return result

raw    = spark.table(RAW_TABLE)
silver = spark.table(SILVER_TABLE)
TABLES = {"raw_schema_layer_table": raw, "silver_schema_layer_table": silver}

## TC-0: Schema / column-name validation
Both tables must contain exactly the expected columns.

In [0]:
# PyDeequ check: a completeness constraint on a column that does not exist fails
# with "Input data does not include column ...", so this validates every expected column is present.
for name, df in TABLES.items():
    check = Check(spark, CheckLevel.Error, "Schema Columns Present")
    for col in EXPECTED_COLUMNS:
        check = check.hasCompleteness(col, lambda x: x >= 0.0, f"Column {col} must exist")
    run_check(df, check, "TC-0 Schema Columns", name)


TC-0 Schema Columns [raw_schema_layer_table] -> Success


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"CompletenessConstraint(Completeness(customer_id,None,None))",Success,
"CompletenessConstraint(Completeness(customer_name,None,None))",Success,
"CompletenessConstraint(Completeness(email,None,None))",Success,
"CompletenessConstraint(Completeness(country,None,None))",Success,
"CompletenessConstraint(Completeness(order_amount,None,None))",Success,
"CompletenessConstraint(Completeness(order_date,None,None))",Success,
"CompletenessConstraint(Completeness(status,None,None))",Success,



TC-0 Schema Columns [silver_schema_layer_table] -> Success


constraint,constraint_status,constraint_message
"CompletenessConstraint(Completeness(customer_id,None,None))",Success,
"CompletenessConstraint(Completeness(customer_name,None,None))",Success,
"CompletenessConstraint(Completeness(email,None,None))",Success,
"CompletenessConstraint(Completeness(country,None,None))",Success,
"CompletenessConstraint(Completeness(order_amount,None,None))",Success,
"CompletenessConstraint(Completeness(order_date,None,None))",Success,
"CompletenessConstraint(Completeness(status,None,None))",Success,


## TC-1: Row count (table is not empty)

In [0]:
for name, df in TABLES.items():
    run_check(df, Check(spark, CheckLevel.Error, "Row Count").hasSize(lambda n: n > 0),
              "TC-1 Row Count", name)


TC-1 Row Count [raw_schema_layer_table] -> Success


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
SizeConstraint(Size(None)),Success,



TC-1 Row Count [silver_schema_layer_table] -> Success


constraint,constraint_status,constraint_message
SizeConstraint(Size(None)),Success,


## TC-2: Completeness (no nulls in mandatory columns)

In [0]:
for name, df in TABLES.items():
    check = (Check(spark, CheckLevel.Error, "Completeness")
        .isComplete("customer_id")
        .isComplete("customer_name")
        .isComplete("order_date")
        .hasCompleteness("email", lambda x: x >= 0.9))
    run_check(df, check, "TC-2 Completeness", name)


TC-2 Completeness [raw_schema_layer_table] -> Error


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"CompletenessConstraint(Completeness(customer_id,None,None))",Failure,Value: 0.8333333333333334 does not meet the constraint requirement!
"CompletenessConstraint(Completeness(customer_name,None,None))",Failure,Value: 0.8333333333333334 does not meet the constraint requirement!
"CompletenessConstraint(Completeness(order_date,None,None))",Failure,Value: 0.8333333333333334 does not meet the constraint requirement!
"CompletenessConstraint(Completeness(email,None,None))",Failure,Value: 0.8333333333333334 does not meet the constraint requirement!



TC-2 Completeness [silver_schema_layer_table] -> Success


constraint,constraint_status,constraint_message
"CompletenessConstraint(Completeness(customer_id,None,None))",Success,
"CompletenessConstraint(Completeness(customer_name,None,None))",Success,
"CompletenessConstraint(Completeness(order_date,None,None))",Success,
"CompletenessConstraint(Completeness(email,None,None))",Success,


## TC-3: Uniqueness (customer_id is a primary key)

In [0]:
for name, df in TABLES.items():
    run_check(df, Check(spark, CheckLevel.Error, "Uniqueness").isUnique("customer_id"),
              "TC-3 Uniqueness", name)


TC-3 Uniqueness [raw_schema_layer_table] -> Error


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"UniquenessConstraint(Uniqueness(List(customer_id),None,None))",Failure,Value: 0.6 does not meet the constraint requirement!



TC-3 Uniqueness [silver_schema_layer_table] -> Success


constraint,constraint_status,constraint_message
"UniquenessConstraint(Uniqueness(List(customer_id),None,None))",Success,


## TC-4: Allowed values (country, status)

In [0]:
for name, df in TABLES.items():
    check = (Check(spark, CheckLevel.Error, "Allowed Values")
        .isContainedIn("country", ["IN", "US", "GB"])
        .isContainedIn("status", ["ACTIVE", "INACTIVE"]))
    run_check(df, check, "TC-4 Allowed Values", name)


TC-4 Allowed Values [raw_schema_layer_table] -> Error


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(country contained in IN,US,GB,`country` IS NULL OR `country` IN ('IN','US','GB'),None,List(country),None))",Failure,Value: 0.8333333333333334 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(status contained in ACTIVE,INACTIVE,`status` IS NULL OR `status` IN ('ACTIVE','INACTIVE'),None,List(status),None))",Failure,Value: 0.8333333333333334 does not meet the constraint requirement!



TC-4 Allowed Values [silver_schema_layer_table] -> Success


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(country contained in IN,US,GB,`country` IS NULL OR `country` IN ('IN','US','GB'),None,List(country),None))",Success,
"ComplianceConstraint(Compliance(status contained in ACTIVE,INACTIVE,`status` IS NULL OR `status` IN ('ACTIVE','INACTIVE'),None,List(status),None))",Success,


## TC-5: Numeric range (order_amount)

In [0]:
for name, df in TABLES.items():
    check = (Check(spark, CheckLevel.Error, "Numeric Range")
        .isNonNegative("order_amount")
        .hasMax("order_amount", lambda m: m <= 10000))
    run_check(df, check, "TC-5 Numeric Range", name)


TC-5 Numeric Range [raw_schema_layer_table] -> Error


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(order_amount is non-negative,COALESCE(CAST(order_amount AS DECIMAL(20,10)), 0.0) >= 0,None,List(order_amount),None))",Failure,Value: 0.8333333333333334 does not meet the constraint requirement!
"MaximumConstraint(Maximum(order_amount,None,None))",Failure,Value: 999999.0 does not meet the constraint requirement!



TC-5 Numeric Range [silver_schema_layer_table] -> Success


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(order_amount is non-negative,COALESCE(CAST(order_amount AS DECIMAL(20,10)), 0.0) >= 0,None,List(order_amount),None))",Success,
"MaximumConstraint(Maximum(order_amount,None,None))",Success,


## TC-6: Email format

In [0]:
EMAIL_REGEX = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
for name, df in TABLES.items():
    check = Check(spark, CheckLevel.Warning, "Email Format").hasPattern("email", EMAIL_REGEX, lambda x: x == 1.0)
    run_check(df, check, "TC-6 Email Format", name)


TC-6 Email Format [raw_schema_layer_table] -> Warning


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"PatternMatchConstraint(email, ^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$)",Failure,Value: 0.6666666666666666 does not meet the constraint requirement!



TC-6 Email Format [silver_schema_layer_table] -> Success


constraint,constraint_status,constraint_message
"PatternMatchConstraint(email, ^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$)",Success,


## TC-7: Raw vs Silver row-count reconciliation

In [0]:
# Get raw row count with a PyDeequ Size analyzer, then assert on silver with a PyDeequ check
raw_metrics = AnalyzerContext.successMetricsAsDataFrame(
    spark, AnalysisRunner(spark).onData(raw).addAnalyzer(Size()).run())
raw_cnt = int(raw_metrics.filter(F.col("name") == "Size").first()["value"])
print(f"raw_schema_layer_table row count = {raw_cnt}")

recon_check = Check(spark, CheckLevel.Error, "Raw vs Silver Reconciliation") \
    .hasSize(lambda n: 0 < n <= raw_cnt, "Silver row count must be > 0 and <= raw row count")
run_check(silver, recon_check, "TC-7 Raw vs Silver Reconciliation", "silver_schema_layer_table")

/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


raw_schema_layer_table row count = 6

TC-7 Raw vs Silver Reconciliation [silver_schema_layer_table] -> Success


constraint,constraint_status,constraint_message
SizeConstraint(Size(None)),Success,


## 5. Summary

In [0]:
from functools import reduce
summary = reduce(DataFrame.unionByName, ALL_RESULTS)
display(summary.select("table", "test", "constraint", "constraint_status", "constraint_message")
               .orderBy("table", "test"))

display(summary.groupBy("table", "constraint_status").count().orderBy("table"))

table,test,constraint,constraint_status,constraint_message
raw_schema_layer_table,TC-0 Schema Columns,"CompletenessConstraint(Completeness(customer_id,None,None))",Success,
raw_schema_layer_table,TC-0 Schema Columns,"CompletenessConstraint(Completeness(customer_name,None,None))",Success,
raw_schema_layer_table,TC-0 Schema Columns,"CompletenessConstraint(Completeness(email,None,None))",Success,
raw_schema_layer_table,TC-0 Schema Columns,"CompletenessConstraint(Completeness(country,None,None))",Success,
raw_schema_layer_table,TC-0 Schema Columns,"CompletenessConstraint(Completeness(order_amount,None,None))",Success,
raw_schema_layer_table,TC-0 Schema Columns,"CompletenessConstraint(Completeness(order_date,None,None))",Success,
raw_schema_layer_table,TC-0 Schema Columns,"CompletenessConstraint(Completeness(status,None,None))",Success,
raw_schema_layer_table,TC-1 Row Count,SizeConstraint(Size(None)),Success,
raw_schema_layer_table,TC-2 Completeness,"CompletenessConstraint(Completeness(customer_id,None,None))",Failure,Value: 0.8333333333333334 does not meet the constraint requirement!
raw_schema_layer_table,TC-2 Completeness,"CompletenessConstraint(Completeness(customer_name,None,None))",Failure,Value: 0.8333333333333334 does not meet the constraint requirement!


table,constraint_status,count
raw_schema_layer_table,Success,8
raw_schema_layer_table,Failure,10
silver_schema_layer_table,Success,19
